
# **ST1519 Agile MLOps CA1 — Shared Chronological Data Split**

## **1. Purpose**

This notebook prepares a shared chronological train-validation-test split for SP-buy fraud detection model training.

All team members will use the same split to ensure fair model comparisons and consistent MLflow evaluation.

**Datasets:**
- `features_base_full.csv`: 41 engineered features, including 10 potentially affected customer snapshot features. Used for exploratory comparison only.
- `features_base_safe.csv`: 31 engineered features, excluding the 10 potentially affected features. Used for final-model development.

Both datasets contain the same historical orders and fraud labels.

**Approach:**
1. Load and verify both processed datasets.
2. Examine the order-date distribution and fraud class distribution over time.
3. Determine chronological training, validation and test boundaries.
4. Verify that the splits contain distinct orders with no overlap.
5. Save reusable splitting logic for all team members.

Following the tutor's recommendation, older orders will be used for training and newer orders for validation and testing.

---

## **2. Load and Verify Data**

### **2.1 Import Libraries and Load Datasets**


In [1]:

from pathlib import Path
import pandas as pd

# Same directory used in PBI 1
DATA_DIR = Path(r"C:/Users/yyuel/OneDrive/Documents/Y2 S2/Agile MLOPs/CA1 data")
PROCESSED_DIR = DATA_DIR / "processed"

# Columns needed for chronological splitting and verification
META_COLS = [
    "country_code",
    "order_id",
    "customer_id",
    "order_date",
    "is_fraud"
]

# Preserve IDs as strings
ID_DTYPES = {
    "country_code": "string",
    "order_id": "string",
    "customer_id": "string"
}

# Load only metadata to verify the datasets
full_meta = pd.read_csv(
    PROCESSED_DIR / "features_base_full.csv",
    usecols=META_COLS,
    dtype=ID_DTYPES
)

safe_meta = pd.read_csv(
    PROCESSED_DIR / "features_base_safe.csv",
    usecols=META_COLS,
    dtype=ID_DTYPES
)

print("Full dataset orders:", len(full_meta))
print("Safe dataset orders:", len(safe_meta))


Full dataset orders: 949872
Safe dataset orders: 949872



### **2.2 Verify Dataset Consistency**

Before splitting, confirm that both datasets contain identical order identifiers, dates and fraud labels.

This ensures the full and restricted feature sets can use exactly the same training, validation and test records, making model comparisons fair.

Also check for missing dates, duplicate order keys and the historical date distribution.


In [2]:

# Verify the same orders and labels are present in both files
assert full_meta[META_COLS].equals(safe_meta[META_COLS]), \
    "The full and safe datasets do not contain identical records!"

# Convert order dates to datetime
safe_meta["order_date"] = pd.to_datetime(
    safe_meta["order_date"],
    errors="raise"
)

# Check date and identifier quality
assert safe_meta["order_date"].notna().all(), \
    "Missing order dates found!"

assert not safe_meta.duplicated(
    ["country_code", "order_id"]
).any(), "Duplicate order keys found!"

assert safe_meta["is_fraud"].isin([0, 1]).all(), \
    "Invalid fraud labels found!"

print("Dataset consistency: PASSED")
print("Total orders:", len(safe_meta))
print("Earliest order:", safe_meta["order_date"].min())
print("Latest order:", safe_meta["order_date"].max())

print("\nOrders by month:")
display(
    safe_meta.groupby(
        safe_meta["order_date"].dt.to_period("M")
    ).size().to_frame("order_count")
)


Dataset consistency: PASSED
Total orders: 949872
Earliest order: 2022-11-12 00:00:00
Latest order: 2023-05-11 00:00:00

Orders by month:


,order_count
order_date,
2022-11,106970
2022-12,171793
2023-01,157820
2023-02,147844
2023-03,159140
2023-04,148855
2023-05,57450



---

## **3. Chronological Split Planning**

### **3.1 Examine Order and Fraud Distribution Over Time**

Before splitting, examine how orders and fraud labels are distributed across different months.

This helps identify whether the fraud rate changes over time and whether the historical data is suitable for chronological model evaluation.

Following the tutor's recommendation, older orders will be used for training and newer orders for validation and testing.

The split will be based on `order_date`, not random sampling.


In [3]:

# Examine monthly order counts and fraud rates
monthly_summary = (
    safe_meta
    .groupby(safe_meta["order_date"].dt.to_period("M"))
    .agg(
        total_orders=("is_fraud", "size"),
        fraud_orders=("is_fraud", "sum")
    )
)

monthly_summary["fraud_rate_pct"] = (
    monthly_summary["fraud_orders"]
    / monthly_summary["total_orders"] * 100
).round(2)

display(monthly_summary)


,total_orders,fraud_orders,fraud_rate_pct
order_date,,,
2022-11,106970,10129,9.47
2022-12,171793,16906,9.84
2023-01,157820,15651,9.92
2023-02,147844,15479,10.47
2023-03,159140,20261,12.73
2023-04,148855,18774,12.61
2023-05,57450,7222,12.57



### **3.2 Determine Train, Validation and Test Boundaries**

A chronological split separates historical orders based on when they were placed.

The proposed proportions are:

- **Training (70%):** Oldest orders, used to fit the machine learning models.
- **Validation (15%):** Subsequent orders, used for hyperparameter tuning, threshold selection and model comparison.
- **Testing (15%):** Most recent orders, reserved for final model evaluation.

These proportions are a starting point rather than a fixed assignment requirement.

To prevent orders from the same calendar date being assigned to different datasets, cutoff dates are selected using cumulative daily order counts.

The same cutoff dates will be used for both the full and restricted feature datasets to ensure fair comparisons.


In [4]:

# Proposed split proportions
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1) < 1e-9

# Count orders per calendar date
order_days = safe_meta["order_date"].dt.normalize()

daily_counts = (
    safe_meta.groupby(order_days)
    .size()
    .sort_index()
)

assert len(daily_counts) >= 3, \
    "At least three distinct order dates are required."

# Cumulative number of orders
cumulative_counts = daily_counts.cumsum()
total_orders = len(safe_meta)

# Target cumulative order counts
train_target = total_orders * TRAIN_RATIO
val_target = total_orders * (TRAIN_RATIO + VAL_RATIO)

# Find the date closest to each target
train_end = cumulative_counts.index[
    (cumulative_counts - train_target).abs().argmin()
]

val_end = cumulative_counts.index[
    (cumulative_counts - val_target).abs().argmin()
]

assert train_end < val_end, \
    "Cutoff dates overlap. Review split ratios."

# Calculate actual numbers of orders
train_count = (order_days <= train_end).sum()

val_count = (
    (order_days > train_end) &
    (order_days <= val_end)
).sum()

test_count = (order_days > val_end).sum()

assert min(train_count, val_count, test_count) > 0, \
    "One of the splits is empty."

# Display proposed boundaries and actual proportions
split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Orders": [train_count, val_count, test_count],
    "Percentage": [
        train_count / total_orders * 100,
        val_count / total_orders * 100,
        test_count / total_orders * 100
    ]
})

split_summary["Percentage"] = (
    split_summary["Percentage"].round(2)
)

print("Train ends:", train_end.date())
print("Validation ends:", val_end.date())
print("Test ends:", order_days.max().date())

display(split_summary)


Train ends: 2023-03-15
Validation ends: 2023-04-13
Test ends: 2023-05-11


,Split,Orders,Percentage
0,Train,665876,70.10
1,Validation,139874,14.73
2,Test,144122,15.17



### **3.3 Create and Validate Chronological Splits**

Using the selected cutoff dates, assign each order to one of three datasets:

- **Training:** Orders placed on or before 15 March 2023
- **Validation:** Orders placed from 16 March to 13 April 2023
- **Testing:** Orders placed after 13 April 2023

The split is based on `order_date`, ensuring older orders are used for training and newer orders for evaluation.

**Validation checks:**
1. Every order belongs to exactly one split.
2. No orders overlap between splits.
3. All 949,872 orders are accounted for.
4. Each split contains both fraudulent and non-fraudulent orders.

The same split boundaries will be applied to the full and restricted datasets.


In [5]:

# Create chronological split masks
order_days = safe_meta["order_date"].dt.normalize()

train_mask = order_days <= train_end

val_mask = (
    (order_days > train_end) &
    (order_days <= val_end)
)

test_mask = order_days > val_end

# Verify every row belongs to exactly one split
assert (train_mask.astype(int) +
        val_mask.astype(int) +
        test_mask.astype(int)).eq(1).all(), \
    "Some orders are missing or assigned to multiple splits!"

# Assign split labels for analysis
split_data = safe_meta.copy()

split_data["split"] = "unassigned"
split_data.loc[train_mask, "split"] = "Train"
split_data.loc[val_mask, "split"] = "Validation"
split_data.loc[test_mask, "split"] = "Test"

# Summarise each split
split_report = (
    split_data.groupby("split")
    .agg(
        total_orders=("is_fraud", "size"),
        fraud_orders=("is_fraud", "sum"),
        start_date=("order_date", "min"),
        end_date=("order_date", "max")
    )
)

split_report["non_fraud_orders"] = (
    split_report["total_orders"] -
    split_report["fraud_orders"]
)

split_report["fraud_rate_pct"] = (
    split_report["fraud_orders"] /
    split_report["total_orders"] * 100
).round(2)

# Final checks
assert len(split_data) == 949872
assert split_report["fraud_orders"].gt(0).all(), \
    "A split has no fraud cases!"

assert split_report["non_fraud_orders"].gt(0).all(), \
    "A split has no non-fraud cases!"

print("Chronological split validation: PASSED")
display(split_report)


Chronological split validation: PASSED


,total_orders,fraud_orders,start_date,end_date,non_fraud_orders,fraud_rate_pct
split,,,,,,
Test,144122,17835,2023-04-14,2023-05-11,126287,12.37
Train,665876,67594,2022-11-12,2023-03-15,598282,10.15
Validation,139874,18993,2023-03-16,2023-04-13,120881,13.58



### **3.4 Verify Consistent Splits Across Both Datasets**

The full and restricted datasets must use identical chronological boundaries to ensure a fair comparison of model performance.

The selected boundaries are:

- **Training:** On or before 15 March 2023
- **Validation:** 16 March to 13 April 2023
- **Testing:** After 13 April 2023

The same splitting function is applied to both datasets.

This ensures that differences in model performance are not caused by different training or evaluation records.

The splitting logic will later be moved into `data_split.py` so all team members can reuse it.


In [6]:

def create_split_labels(df, train_end, val_end):
    """Assign Train, Validation or Test using order dates."""
    dates = pd.to_datetime(
        df["order_date"], errors="raise"
    ).dt.normalize()

    assert dates.notna().all(), "Missing order dates found."
    assert train_end < val_end, "Invalid cutoff dates."

    labels = pd.Series(index=df.index, dtype="string")

    labels.loc[dates <= train_end] = "Train"

    labels.loc[
        (dates > train_end) & (dates <= val_end)
    ] = "Validation"

    labels.loc[dates > val_end] = "Test"

    assert labels.notna().all(), "Unassigned orders found."

    return labels


# Apply identical boundaries to both datasets
full_splits = create_split_labels(
    full_meta, train_end, val_end
)

safe_splits = create_split_labels(
    safe_meta, train_end, val_end
)

# Verify both files receive identical split assignments
assert full_splits.equals(safe_splits), \
    "Full and safe datasets have different split assignments!"

# Verify expected record counts
assert safe_splits.value_counts().to_dict() == {
    "Train": 665876,
    "Validation": 139874,
    "Test": 144122
}, "Unexpected split counts!"

print("Both datasets use identical chronological splits: PASSED")

display(
    safe_splits.value_counts()
    .reindex(["Train", "Validation", "Test"])
    .to_frame("orders")
)


Both datasets use identical chronological splits: PASSED


,orders
Train,665876
Validation,139874
Test,144122



---

## **4. Test Reusable Data Splitting Module**

### **4.1 Import and Verify `data_split.py`**

The chronological splitting logic has been moved into `src/data_split.py` so all team members can use the same training, validation and test boundaries.

This section verifies that:

1. The Python module can be imported successfully.
2. The module uses the agreed chronological cutoff dates.
3. Both feature datasets receive identical split assignments.
4. The module produces the same results as the splitting logic tested in Section 3.4.

This ensures that the shared splitting function is consistent and ready for model training.


In [7]:

from pathlib import Path
import sys
import importlib

# Locate the repository containing src/data_split.py
repo_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "src" / "data_split.py").is_file()
    ),
    None
)

assert repo_root is not None, "Cannot find src/data_split.py"

# Allow the notebook to import from src/
src_path = str(repo_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

import data_split
importlib.reload(data_split)

print("Imported from:", data_split.__file__)

# Check the shared cutoff dates
assert data_split.TRAIN_END == train_end
assert data_split.VAL_END == val_end

# Apply the imported function to both datasets
full_module_splits = data_split.create_split_labels(full_meta)
safe_module_splits = data_split.create_split_labels(safe_meta)

# Compare with previously verified notebook results
assert full_module_splits.equals(full_splits)
assert safe_module_splits.equals(safe_splits)

# Confirm both datasets use identical assignments
assert full_module_splits.equals(safe_module_splits)

print("\nReusable splitting module: PASSED")
print("Train ends:", data_split.TRAIN_END.date())
print("Validation ends:", data_split.VAL_END.date())

display(
    safe_module_splits.value_counts()
    .reindex(["Train", "Validation", "Test"])
    .to_frame("orders")
)


Imported from: c:\Users\yyuel\OneDrive\Documents\Y2 S2\Agile MLOPs\MLOps_CA1_group5\src\data_split.py

Reusable splitting module: PASSED
Train ends: 2023-03-15
Validation ends: 2023-04-13


,orders
Train,665876
Validation,139874
Test,144122



---

## **5. Summary and Model Training Guidelines**

### **5.1 Final Chronological Split**

A shared chronological train-validation-test split was created using `order_date`, following the tutor's recommendation.

| Split | Date Range | Orders | Percentage | Fraud Rate |
|---|---|---:|---:|---:|
| Training | 12 Nov 2022 – 15 Mar 2023 | 665,876 | 70.10% | 10.15% |
| Validation | 16 Mar – 13 Apr 2023 | 139,874 | 14.73% | 13.58% |
| Testing | 14 Apr – 11 May 2023 | 144,122 | 15.17% | 12.37% |

### **5.2 Validation Results**

- All **949,872 orders** were assigned to exactly one split.
- No order overlaps were found between training, validation and testing.
- Orders placed on the same calendar date remain in the same split.
- Both the full (41 features) and restricted (31 features) datasets use identical split assignments.
- Each split contains both fraudulent and non-fraudulent orders.
- Fraud rates differ across periods, indicating that the distribution of fraud changes over time.
- The reusable `src/data_split.py` module was imported and tested successfully.

### **5.3 Guidelines for Model Training**

- All team members must use the shared `data_split.py` module for consistent model comparisons.
- Use the **training set** to fit models.
- Use the **validation set** for hyperparameter tuning, classification threshold selection and model comparison.
- Keep the **test set untouched** until final model evaluation.
- Exclude `country_code`, `order_id`, `customer_id`, `order_date` and `is_fraud` from prediction features.
- Use `is_fraud` as the target variable.
- Use `features_base_safe.csv` for final-model candidates. The full dataset is only for exploratory comparison of potentially leaked customer snapshot features.
- Any scaling, resampling or other fitted preprocessing must be learned from the training data only.
- Prioritise **high precision with reasonable recall**, as required by the Product Owner.

### **5.4 Reusability**

The shared splitting logic is stored in `src/data_split.py`, allowing every team member to apply the same historical cutoff dates without manually recreating the splits.

These dates apply to the current historical dataset. Future automated retraining may require updated chronological boundaries.

---
